# 08 — Model Evaluation

**Goals**
- Collect LOLO predictions for every model
- Accuracy + confusion matrices, split by indoor / outdoor
- Save plots and tables to `results/`


In [ ]:
import pandas as pd
import json
import joblib
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

PROCESSED_DIR = Path('../data/processed')
MODELS_DIR = Path('../models')
RESULTS_DIR = Path('../results')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(PROCESSED_DIR / 'labeled_dataset.csv')
with open(PROCESSED_DIR / 'lolo_folds.json') as f:
    folds = json.load(f)
le = joblib.load(MODELS_DIR / 'label_encoder.joblib')
fold_models = joblib.load(MODELS_DIR / 'fold_models.joblib')

FEATURE_COLS = [
    'temperature_C', 'humidity_pct', 'soil_moisture_pct',
    'moisture_trend', 'light_lux', 'hour_of_day'
]
print('Classes:', list(le.classes_))


In [ ]:
# Collect predictions across all folds
records = []
for fold in folds:
    fid = fold['fold_id']
    test_mask = df.cycle_id.isin(fold['test_cycles'])
    X_test = df.loc[test_mask, FEATURE_COLS]
    y_true = df.loc[test_mask, 'urgency']
    cond = fold['condition']

    for name in ['logistic', 'decision_tree', 'random_forest']:
        if name not in fold_models.get(fid, {}):
            pred = ['<24h'] * len(X_test)   # fallback for skipped logistic
        else:
            pred_enc = fold_models[fid][name].predict(X_test)
            pred = le.inverse_transform(pred_enc)
        for yt, yp in zip(y_true, pred):
            records.append({
                'fold': fid, 'condition': cond, 'model': name,
                'y_true': yt, 'y_pred': yp
            })

preds_df = pd.DataFrame(records)
preds_df.groupby(['condition', 'model']).size()


In [ ]:
# Accuracy & classification reports
results = []
for (cond, model), g in preds_df.groupby(['condition', 'model']):
    acc = accuracy_score(g['y_true'], g['y_pred'])
    results.append({'condition': cond, 'model': model, 'accuracy': acc, 'n': len(g)})
    print(f'\n=== {model} | {cond} | acc={acc:.3f} ===')
    print(classification_report(g['y_true'], g['y_pred'], zero_division=0))

res_df = pd.DataFrame(results)
print('\nSummary table:')
display(res_df)
res_df.to_csv(RESULTS_DIR / '08_lolo_accuracy.csv', index=False)


In [ ]:
# Confusion-matrix grid
fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for i, cond in enumerate(['indoor', 'outdoor']):
    for j, model in enumerate(['logistic', 'decision_tree', 'random_forest']):
        ax = axes[i, j]
        g = preds_df[(preds_df.condition == cond) & (preds_df.model == model)]
        cm = confusion_matrix(g['y_true'], g['y_pred'], labels=list(le.classes_))
        ax.imshow(cm, cmap='Blues')
        ax.set_xticks(range(len(le.classes_)))
        ax.set_yticks(range(len(le.classes_)))
        ax.set_xticklabels(le.classes_, rotation=45, ha='right')
        ax.set_yticklabels(le.classes_)
        ax.set_title(f'{model}\n{cond}')
        for r in range(cm.shape[0]):
            for c in range(cm.shape[1]):
                ax.text(c, r, cm[r, c], ha='center', va='center')
plt.tight_layout()
plt.savefig(RESULTS_DIR / '08_confusion_matrices.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved → results/08_confusion_matrices.png')
